# Preprocessing LSTM inputs

In [ ]:
# general
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import yaml 
import os

# geospatial
import geopandas as gpd
import xarray as xr

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd) 

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

In [ ]:
# Create output directories
output_dir = Path(cfg["dir"]["pipeline"] + "/lstm/data")
(output_dir / "time_series").mkdir(parents=True, exist_ok=True)
(output_dir / "attributes").mkdir(parents=True, exist_ok=True)

## Processing attributes

In [ ]:
# 1. ATTRIBUTES - Save as CSV in attributes folder
metadata_ggc = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "dataset/AndeanGC_metadata.csv", index_col=0)
metadata_all = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").drop(columns=["geometry"]).set_index("gauge_id")
metadata = pd.concat([metadata_ggc, metadata_all])

# Normalization
metadata["glacier_area_RGI60"] = metadata.glacier_area_RGI60 / metadata.basin_area
metadata["glacier_volume_M22"] = metadata.glacier_volume_M22 / metadata.basin_area

# Filter basins with glaciers
metadata = metadata[metadata.glacier_area_RGI60 > 0]

# Save attributes as CSV (indexed by gauge_id)
metadata = metadata.dropna(axis = 1)
metadata.to_csv(output_dir / "attributes" / "basin_attributes.csv")

## Processing time series

In [ ]:
# streamflow data -> DataArray with dims ('date', 'basin_id')
data = pd.read_csv(cfg["dir"]["andeangc"] + "/dataset/AndeanGC_data_1950_2024_qc.csv", index_col=0, parse_dates=True)
data = data.loc[cfg["time"]["ref_start"]:cfg["time"]["ref_end"]]
data = (data * 1000 * 86400) / (metadata.basin_area * 1e6)

data = xr.DataArray(
    data.values,
    coords={"date": data.index, "basin_id": data.columns},
    dims=("date", "basin_id"),
    name="q_mm")

In [ ]:
# historical climate (ERA5)
vars = ["tasmax", "tasmin", "tas", "ep", "prcp"]

for var in tqdm(vars):
    climate_data_gcc = pd.read_parquet(cfg["dir"]["andeangc"] + "climate/historical/AndeanGC_{}_ERA5_1960_2024.parquet".format(var))
    climate_data_all = pd.read_parquet("data/climate/HydroAtlas_{}_ERA5_1990_2024.parquet".format(var))
    climate_data = pd.concat([climate_data_gcc, climate_data_all], axis=1)

    climate_data = climate_data.loc[cfg["time"]["ref_start"]:cfg["time"]["ref_end"]]
    climate_data = climate_data.reset_index().set_index("time")
    climate_data.index.name = "date"
    climate_data = climate_data[metadata.index]
    climate_data = xr.DataArray(
        climate_data.values,
        coords={"date": climate_data.index, "basin_id": climate_data.columns},
        dims=("date", "basin_id"),
        name=var)
    data = xr.merge([data, climate_data])

In [ ]:
# OGGM glacier melt (historical + what-if)
paths = Path("data/glacier//").glob("*historical*.nc")

for path in tqdm(paths):
    name = path.name.replace("run_output_", "gmelt_").replace(".nc", "")

    q_glacier = xr.open_dataset(path).glacier_melt
    q_glacier = q_glacier.reindex(gauge_id=metadata.index, fill_value=0)
    q_glacier = (q_glacier*1000*86400) / (metadata.basin_area.values*1e6)[:, None]
    q_glacier = q_glacier.rename({"gauge_id": "basin_id"})

    q_glacier = q_glacier.rename(name)
    data = xr.merge([data, q_glacier])

In [ ]:
for basin in tqdm(data.basin_id.values):
    data_basin = data.sel(basin_id=basin)
    data_basin = data_basin.astype('float32')
    data_basin.to_netcdf(output_dir / "time_series" / "{}.nc".format(basin), 
                         encoding={var: {"compression": "zstd"} for var in data_basin.data_vars})